<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/Detector_de_Somnolencia_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/laboratoriodecodigos/Colab-Python/blob/main/Detector_de_Somnolencia_MediaPipe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Esta IA te despierta si cierras los ojos 2 segundos
**Laboratorio de Códigos** · MediaPipe Face Landmarker + EAR + PERCLOS

Genera, además del detector:
- 🎬 **Video vertical 1080×1920** listo para TikTok: título, gráfica en vivo, contadores y **alarma sonora**.
- 📊 **Datos**: `datos_por_cuadro.csv`, `eventos.csv` y `resumen.json` con los números para el guion.
- 🖼️ **Gráficas**: la curva de apertura de ojos y una tarjeta de resultados para el cierre.

In [ ]:
!pip install -q mediapipe
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task

In [ ]:
from google.colab import files
subido = files.upload()
VIDEO_IN = next(iter(subido))     # usa el nombre real del archivo subido
print("Video:", VIDEO_IN)

Saving video somnolencia.mp4 to video somnolencia.mp4
Video: video somnolencia.mp4


In [ ]:
# ⚙️ Ajustes
SEG_ALERTA      = 2.0    # segundos con ojos cerrados para activar la alarma
SEG_PARPADEO    = 0.4    # un cierre más corto que esto es un parpadeo normal
SEG_CALIBRACION = 3.0    # primeros segundos: mide tus ojos ABIERTOS (mira a la cámara)
FACTOR_UMBRAL   = 0.75   # umbral = EAR con ojos abiertos × este factor
VENTANA_PERCLOS = 30     # segundos para calcular PERCLOS (% del tiempo con ojos cerrados)

TITULO = "Esta IA te despierta si cierras los ojos 2 segundos"


## 1. Medir la apertura de los ojos (EAR) en cada cuadro

In [ ]:
import cv2, time, numpy as np, mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

LEFT_EYE  = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]

def EAR(p):
    """Eye Aspect Ratio: alto del ojo / ancho del ojo. Abierto ≈ 0.25-0.35, cerrado ≈ 0.1"""
    A = np.linalg.norm(p[1] - p[5])
    B = np.linalg.norm(p[2] - p[4])
    C = np.linalg.norm(p[0] - p[3])
    return (A + B) / (2 * C)

# Modo VIDEO: MediaPipe usa los cuadros anteriores y el seguimiento es más estable
detector = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="face_landmarker.task"),
    running_mode=vision.RunningMode.VIDEO, num_faces=1))

cap = cv2.VideoCapture(VIDEO_IN)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
W, H = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

ears, ojos, ms = [], [], []
i = 0
while True:
    ok, frame = cap.read()
    if not ok:
        break
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    t0 = time.perf_counter()
    res = detector.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), int(i * 1000 / fps))
    ms.append((time.perf_counter() - t0) * 1000)
    if res.face_landmarks:
        pts = np.array([[p.x * W, p.y * H] for p in res.face_landmarks[0]])
        izq, der = pts[LEFT_EYE], pts[RIGHT_EYE]
        ears.append((EAR(izq) + EAR(der)) / 2)
        ojos.append(np.vstack([izq, der]))
    else:
        ears.append(np.nan)
        ojos.append(None)
    i += 1
cap.release(); detector.close()
print(f"{len(ears)} cuadros a {fps:.0f} fps · {np.median(ms):.1f} ms por cuadro · rostro en {np.isfinite(ears).mean()*100:.0f}% del video")

1297 cuadros a 60 fps · 19.8 ms por cuadro · rostro en 96% del video


## 2. Calibrar, detectar eventos y calcular métricas
- **Umbral personal:** se calcula con tus primeros segundos con ojos abiertos (no un 0.22 fijo).
- **Parpadeo** < 0.4 s · **cierre largo** entre 0.4 y 2 s · **microsueño** ≥ 2 s → alarma.
- **PERCLOS:** % del tiempo con ojos cerrados en los últimos 30 s, una medida usada en estudios de fatiga al volante.

In [ ]:
import pandas as pd

N = len(ears)
t = np.arange(N) / fps

# 1) Suavizado: rellena huecos cortos sin rostro (0.5 s) y quita ruido de 1 cuadro
ear = pd.Series(ears, dtype=float).ffill(limit=int(0.5 * fps)).rolling(3, center=True, min_periods=1).median().to_numpy()

# 2) Calibración personal: cada persona tiene ojos distintos, así que el umbral se adapta
n_cal = int(SEG_CALIBRACION * fps)
if np.isfinite(ear[:n_cal]).sum() > fps:
    EAR_ABIERTO = float(np.nanmedian(ear[:n_cal]))
else:
    EAR_ABIERTO = float(np.nanpercentile(ear, 90))
UMBRAL = EAR_ABIERTO * FACTOR_UMBRAL
cerrado = np.nan_to_num(ear, nan=np.inf) < UMBRAL

# 3) Eventos: cada racha de ojos cerrados es un parpadeo, un cierre largo o un microsueño
eventos, i = [], 0
while i < N:
    if cerrado[i]:
        j = i
        while j < N and cerrado[j]:
            j += 1
        dur = (j - i) / fps
        tipo = "microsueño" if dur >= SEG_ALERTA else ("parpadeo" if dur < SEG_PARPADEO else "cierre largo")
        eventos.append({"inicio_s": round(i / fps, 2), "fin_s": round(j / fps, 2), "duracion_s": round(dur, 2),
                        "tipo": tipo, "alarma_s": round(i / fps + SEG_ALERTA, 2) if tipo == "microsueño" else None,
                        "_i": i, "_j": j})
        i = j
    else:
        i += 1
ev = pd.DataFrame(eventos, columns=["inicio_s", "fin_s", "duracion_s", "tipo", "alarma_s", "_i", "_j"])

# 4) Series por cuadro para el video
seg_cerrado = np.zeros(N)
for k in range(N):
    seg_cerrado[k] = (seg_cerrado[k - 1] + 1 / fps) if (cerrado[k] and k > 0) else (1 / fps if cerrado[k] else 0)
alerta = seg_cerrado >= SEG_ALERTA
perclos = pd.Series(cerrado.astype(float)).rolling(int(VENTANA_PERCLOS * fps), min_periods=1).mean().to_numpy() * 100

marca_parpadeo, marca_micro = np.zeros(N), np.zeros(N)
for _, e in ev.iterrows():
    if e.tipo == "parpadeo":
        marca_parpadeo[min(int(e._j), N - 1)] += 1
    elif e.tipo == "microsueño":
        marca_micro[min(int(e._i + SEG_ALERTA * fps), N - 1)] += 1
parpadeos_acum, micro_acum = np.cumsum(marca_parpadeo), np.cumsum(marca_micro)

# 5) Resumen: los números para el guion y la portada
dur_total = N / fps
micro = ev[ev.tipo == "microsueño"]
resumen = {
    "duracion_video_s": round(dur_total, 1),
    "ear_ojos_abiertos": round(EAR_ABIERTO, 3),
    "umbral_ear": round(UMBRAL, 3),
    "parpadeos": int((ev.tipo == "parpadeo").sum()),
    "parpadeos_por_minuto": round((ev.tipo == "parpadeo").sum() / (dur_total / 60), 1),
    "cierres_largos": int((ev.tipo == "cierre largo").sum()),
    "microsuenos": int(len(micro)),
    "primera_alarma_s": float(micro.alarma_s.iloc[0]) if len(micro) else None,
    "microsueno_mas_largo_s": float(micro.duracion_s.max()) if len(micro) else 0.0,
    "tiempo_total_ojos_cerrados_s": round(cerrado.sum() / fps, 1),
    "perclos_max_pct": round(float(np.max(perclos)), 1),
    "ms_por_cuadro": round(float(np.median(ms)), 1) if "ms" in globals() and len(ms) else None,
    "cuadros_sin_rostro_pct": round(float(np.mean(~np.isfinite(np.array(ears, dtype=float))) * 100), 1),
}

datos = pd.DataFrame({"t_s": t.round(3), "ear": np.round(ear, 4), "umbral": round(UMBRAL, 4), "ojos_cerrados": cerrado,
                      "seg_cerrado": seg_cerrado.round(2), "alarma": alerta, "perclos_pct": perclos.round(1)})
datos.to_csv("datos_por_cuadro.csv", index=False)
ev.drop(columns=["_i", "_j"]).to_csv("eventos.csv", index=False)
import json
json.dump(resumen, open("resumen.json", "w"), ensure_ascii=False, indent=2)

for k, v in resumen.items():
    print(f"{k:<30} {v}")
print()
print(ev.drop(columns=["_i", "_j"]).to_string(index=False))

duracion_video_s               21.6
ear_ojos_abiertos              0.25
umbral_ear                     0.187
parpadeos                      3
parpadeos_por_minuto           8.3
cierres_largos                 1
microsuenos                    2
primera_alarma_s               4.83
microsueno_mas_largo_s         2.68
tiempo_total_ojos_cerrados_s   7.5
perclos_max_pct                49.4
ms_por_cuadro                  19.8
cuadros_sin_rostro_pct         3.9

 inicio_s  fin_s  duracion_s         tipo  alarma_s
     0.55   0.73        0.18     parpadeo       NaN
     2.83   5.23        2.40   microsueño      4.83
     8.25  10.93        2.68   microsueño     10.25
    13.30  13.47        0.17     parpadeo       NaN
    15.10  15.42        0.32     parpadeo       NaN
    16.60  18.37        1.77 cierre largo       NaN


## 3. Video vertical para TikTok

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import matplotlib, os, textwrap

FUENTE = os.path.join(matplotlib.get_data_path(), "fonts", "ttf", "DejaVuSans-Bold.ttf")
f_titulo, f_stat, f_peq, f_alerta = (ImageFont.truetype(FUENTE, s) for s in (62, 44, 30, 70))
CW, CH = 1080, 1920                                   # lienzo vertical TikTok
FONDO, AMARILLO, ROJO, VERDE, GRIS = (15, 17, 26), (255, 214, 10), (230, 40, 40), (60, 220, 120), (150, 150, 160)

# Base fija: título arriba
base = Image.new("RGB", (CW, CH), FONDO)
d = ImageDraw.Draw(base)
d.text((50, 40), "LABORATORIO DE CÓDIGOS · MediaPipe", font=f_peq, fill=GRIS)
y = 100
for linea in textwrap.wrap(TITULO, 26):
    d.text((50, y), linea, font=f_titulo, fill=(255, 255, 255)); y += 76

# Dónde va el video
escala = min(CW / W, 900 / H)
VW, VH = int(W * escala), int(H * escala)
VX, VY = (CW - VW) // 2, max(y + 30, 360)
GY = VY + VH + 30                                     # gráfica en vivo
SY = GY + 290                                         # estadísticas

def grafica(k, ancho=980, alto=250, seg=6):
    """EAR de los últimos `seg` segundos con la línea del umbral."""
    g = np.full((alto, ancho, 3), (32, 26, 22), np.uint8)    # BGR
    a, b = max(0, k - int(seg * fps)), k + 1
    ymax = max(EAR_ABIERTO * 1.4, 0.05)
    Y = lambda v: int(alto - 20 - (v / ymax) * (alto - 40))
    cv2.line(g, (0, Y(UMBRAL)), (ancho, Y(UMBRAL)), (10, 214, 255), 2)
    vals = ear[a:b]
    xs = np.linspace(0, ancho - 1, int(seg * fps) + 1)[-len(vals):]
    pts = [(int(x), Y(v)) for x, v in zip(xs, vals) if np.isfinite(v)]
    col = (40, 40, 230) if alerta[k] else ((40, 140, 255) if cerrado[k] else (120, 220, 60))
    if len(pts) > 1:
        cv2.polylines(g, [np.array(pts, np.int32)], False, col, 4)
    return Image.fromarray(g[:, :, ::-1])

cap = cv2.VideoCapture(VIDEO_IN)
out = cv2.VideoWriter("somnolencia_raw.mp4", cv2.VideoWriter_fourcc(*"mp4v"), fps, (CW, CH))
for k in range(N):
    ok, frame = cap.read()
    if not ok:
        break
    color_ojos = (0, 0, 255) if alerta[k] else ((0, 165, 255) if cerrado[k] else (0, 255, 0))
    if ojos[k] is not None:
        for p in ojos[k]:
            cv2.circle(frame, (int(p[0]), int(p[1])), max(2, W // 300), color_ojos, -1)
    lienzo = base.copy()
    lienzo.paste(Image.fromarray(cv2.resize(frame, (VW, VH))[:, :, ::-1]), (VX, VY))
    d = ImageDraw.Draw(lienzo)

    if alerta[k]:
        if int(t[k] * 4) % 2 == 0:                    # parpadea 2 veces por segundo
            d.rectangle([0, 0, CW - 1, CH - 1], outline=ROJO, width=24)
        d.rectangle([VX, VY, VX + VW, VY + 110], fill=ROJO)
        d.text((VX + 30, VY + 18), "¡DESPIERTA!", font=f_alerta, fill=(255, 255, 255))
    else:
        etiqueta, col = ("OJOS CERRADOS", AMARILLO) if cerrado[k] else ("DESPIERTO", VERDE)
        d.rectangle([VX, VY, VX + 420, VY + 70], fill=(0, 0, 0))
        d.text((VX + 20, VY + 14), etiqueta, font=f_peq, fill=col)

    lienzo.paste(grafica(k), (50, GY))
    d.text((60, GY + 8), f"EAR {ear[k]:.2f}" if np.isfinite(ear[k]) else "sin rostro", font=f_peq, fill=(255, 255, 255))
    d.text((CW - 330, GY + 8), f"umbral {UMBRAL:.2f}", font=f_peq, fill=AMARILLO)

    col_t = ROJO if alerta[k] else (AMARILLO if cerrado[k] else (255, 255, 255))
    d.text((50, SY), f"Ojos cerrados: {seg_cerrado[k]:.1f} s / {SEG_ALERTA:.0f} s", font=f_stat, fill=col_t)
    d.text((50, SY + 70), f"Parpadeos: {int(parpadeos_acum[k])}", font=f_stat, fill=(255, 255, 255))
    d.text((560, SY + 70), f"Microsueños: {int(micro_acum[k])}", font=f_stat, fill=ROJO if micro_acum[k] else (255, 255, 255))
    d.text((50, SY + 140), f"PERCLOS: {perclos[k]:.0f}%", font=f_stat, fill=(255, 255, 255))
    d.text((560, SY + 148), f"t = {t[k]:.1f} s", font=f_peq, fill=GRIS)
    out.write(cv2.cvtColor(np.array(lienzo), cv2.COLOR_RGB2BGR))
cap.release(); out.release()
print("Video vertical listo")

Video vertical listo


In [ ]:
# 🔊 Alarma sonora en cada microsueño (beep intermitente) + video final
import wave
SR = 44100
audio = np.zeros(int(SR * N / fps), np.float32)
for _, e in ev[ev.tipo == "microsueño"].iterrows():
    for ini in np.arange(e.alarma_s, e.fin_s, 0.3):
        a, b = int(ini * SR), min(int((ini + 0.15) * SR), len(audio))
        if a < b:
            audio[a:b] = 0.5 * np.sin(2 * np.pi * 880 * np.arange(b - a) / SR)
with wave.open("alarma.wav", "wb") as w:
    w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR)
    w.writeframes((audio * 32767).astype(np.int16).tobytes())

SALIDA = "somnolencia_tiktok.mp4"
!ffmpeg -y -loglevel error -i somnolencia_raw.mp4 -i alarma.wav -c:v libx264 -preset veryfast -pix_fmt yuv420p -c:a aac -shortest {SALIDA}

from base64 import b64encode
from IPython.display import HTML
datos_b64 = b64encode(open(SALIDA, "rb").read()).decode()
HTML(f'<video width=360 controls autoplay loop><source src="data:video/mp4;base64,{datos_b64}" type="video/mp4"></video>')

**Para ajustar:** si marca ojos cerrados con los ojos abiertos, baja `FACTOR_UMBRAL` a 0.7; si no detecta cierres, súbelo a 0.8. Con lentes o luz lateral conviene grabar con buena iluminación de frente.